Задача 3. Плотва или краснопёрка

Нужно по фото понять, какая рыба: плотва или краснопёрка. Выбрал их, потому что они очень похожи, их путают даже рыбаки.

20 фото с Wikimedia Commons: 10 плотвы и 10 краснопёрки, правильный ответ в названии файла. Есть сложные фото: рыба в руке, под водой, на крючке, рядом с другой рыбой.

Модели: GLM-4.6V-Flash (провайдер Z.ai) и Qwen2.5-VL-72B-Instruct (провайдер OVHcloud). На бесплатном аккаунте Hugging Face нет кредитов, поэтому обращаюсь к провайдерам напрямую: у Z.ai эта модель бесплатная, OVHcloud пускает без ключа.

Считаю общую точность и точность отдельно по каждой рыбе.

In [1]:
import os
import time
import base64
import pandas as pd
from huggingface_hub import InferenceClient

zai_key = "0717bc7eb22c4122981f0602165cd349.I2ucQXgTih24ZBhO"

client1 = InferenceClient(provider="zai-org", api_key=zai_key)
model1 = "zai-org/GLM-4.6V-Flash"

client2 = InferenceClient(base_url="https://oai.endpoints.kepler.ai.cloud.ovh.net/v1")
model2 = "Qwen2.5-VL-72B-Instruct"

In [2]:
files = os.listdir("images")
print("Всего фото:", len(files))
print(files)

Всего фото: 20
['krasnoperka_01.jpg', 'krasnoperka_02.jpg', 'krasnoperka_03.jpg', 'krasnoperka_04.jpg', 'krasnoperka_05.jpg', 'krasnoperka_06.jpg', 'krasnoperka_07.jpg', 'krasnoperka_08.jpg', 'krasnoperka_09.jpg', 'krasnoperka_10.jpg', 'plotva_01.jpg', 'plotva_02.jpg', 'plotva_03.jpg', 'plotva_04.jpg', 'plotva_05.jpg', 'plotva_06.jpg', 'plotva_07.jpg', 'plotva_08.jpg', 'plotva_09.jpg', 'plotva_10.jpg']


In [3]:
question = "Какая рыба на фото: плотва или краснопёрка? Ответь одним словом."

def ask(client, model, file):
    f = open("images/" + file, "rb")
    picture = base64.b64encode(f.read()).decode()
    f.close()

    message = [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + picture}},
                {"type": "text", "text": question}
            ]
        }
    ]

    response = client.chat_completion(model=model, messages=message, max_tokens=1000)
    answer = response.choices[0].message.content
    print(model, ":", answer)
    answer = answer.lower()

    if "красноп" in answer:
        return "krasnoperka"
    elif "плотв" in answer:
        return "plotva"
    else:
        return "neponyatno"

In [4]:
table = []

for file in files:
    pravilno = file.split("_")[0]
    otvet1 = ask(client1, model1, file)
    otvet2 = ask(client2, model2, file)
    print(file, pravilno, otvet1, otvet2)
    table.append([file, pravilno, otvet1, otvet2])
    time.sleep(60)
df = pd.DataFrame(table, columns=["fail", "pravilno", "glm", "qwen"])
df

zai-org/GLM-4.6V-Flash : 
краснопёрка
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_01.jpg krasnoperka krasnoperka krasnoperka
zai-org/GLM-4.6V-Flash : 
краснопёрка
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_02.jpg krasnoperka krasnoperka krasnoperka
zai-org/GLM-4.6V-Flash : 
плотва
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_03.jpg krasnoperka plotva krasnoperka
zai-org/GLM-4.6V-Flash : 
плотва
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_04.jpg krasnoperka plotva krasnoperka
zai-org/GLM-4.6V-Flash : 
краснопёрка
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_05.jpg krasnoperka krasnoperka krasnoperka
zai-org/GLM-4.6V-Flash : 
краснопёрка
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_06.jpg krasnoperka krasnoperka krasnoperka
zai-org/GLM-4.6V-Flash : 
плотва
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_07.jpg krasnoperka plotva krasnoperka
zai-org/GLM-4.6V-Flash : 
краснопёрка
Qwen2.5-VL-72B-Instruct : Краснопёрка
krasnoperka_08.jpg krasnoperka krasno

,fail,pravilno,glm,qwen
0,krasnoperka_01.jpg,krasnoperka,krasnoperka,krasnoperka
1,krasnoperka_02.jpg,krasnoperka,krasnoperka,krasnoperka
2,krasnoperka_03.jpg,krasnoperka,plotva,krasnoperka
3,krasnoperka_04.jpg,krasnoperka,plotva,krasnoperka
4,krasnoperka_05.jpg,krasnoperka,krasnoperka,krasnoperka
5,krasnoperka_06.jpg,krasnoperka,krasnoperka,krasnoperka
6,krasnoperka_07.jpg,krasnoperka,plotva,krasnoperka
7,krasnoperka_08.jpg,krasnoperka,krasnoperka,krasnoperka
8,krasnoperka_09.jpg,krasnoperka,neponyatno,krasnoperka
9,krasnoperka_10.jpg,krasnoperka,krasnoperka,krasnoperka


In [5]:
def tochnost(nomer, ryba):
    vsego = 0
    ugadal = 0
    for stroka in table:
        if ryba == "vse" or stroka[1] == ryba:
            vsego = vsego + 1
            if stroka[nomer] == stroka[1]:
                ugadal = ugadal + 1
    return round(ugadal / vsego * 100)

itog = []
itog.append(["GLM", tochnost(2, "vse"), tochnost(2, "krasnoperka"), tochnost(2, "plotva")])
itog.append(["Qwen", tochnost(3, "vse"), tochnost(3, "krasnoperka"), tochnost(3, "plotva")])

itog = pd.DataFrame(itog, columns=["model", "obshaya %", "krasnoperka %", "plotva %"])
itog

,model,obshaya %,krasnoperka %,plotva %
0,GLM,60,60,60
1,Qwen,65,100,30


In [6]:
for stroka in table:
    if stroka[2] != stroka[1] or stroka[3] != stroka[1]:
        print(stroka)

['krasnoperka_03.jpg', 'krasnoperka', 'plotva', 'krasnoperka']
['krasnoperka_04.jpg', 'krasnoperka', 'plotva', 'krasnoperka']
['krasnoperka_07.jpg', 'krasnoperka', 'plotva', 'krasnoperka']
['krasnoperka_09.jpg', 'krasnoperka', 'neponyatno', 'krasnoperka']
['plotva_01.jpg', 'plotva', 'plotva', 'krasnoperka']
['plotva_04.jpg', 'plotva', 'plotva', 'krasnoperka']
['plotva_05.jpg', 'plotva', 'neponyatno', 'krasnoperka']
['plotva_06.jpg', 'plotva', 'neponyatno', 'plotva']
['plotva_07.jpg', 'plotva', 'neponyatno', 'krasnoperka']
['plotva_08.jpg', 'plotva', 'plotva', 'krasnoperka']
['plotva_09.jpg', 'plotva', 'krasnoperka', 'krasnoperka']
['plotva_10.jpg', 'plotva', 'plotva', 'krasnoperka']


Вывод

Qwen угадала 65% фото, GLM 60%, по общей точности модели почти одинаковые, Qwen чуть лучше.

Но ошибаются они по-разному. Qwen почти всё называет краснопёркой: краснопёрку угадала всю, а плотву только 3 из 10. GLM угадала по 60% каждой рыбы, но на 4 фото вообще ничего не ответила (пустой ответ), это засчитано как ошибка.

Получается, обе модели плохо различают этих рыб. По одной общей точности этого не видно, поэтому полезно считать точность по каждой рыбе.